In [ ]:
# ============================================================
# INTERVIEW PREP DATASET GENERATOR
# Student only changes these 2 variables
# ============================================================

CATALOG = "your_catalog"
SCHEMA = "your_schema"

# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# ============================================================
# USERS
# ============================================================

users = (
    spark.range(1, 5001)
    .withColumnRenamed("id", "user_id")
    .withColumn(
        "signup_date",
        F.date_sub(F.current_date(), (F.rand()*730).cast("int"))
    )
)

users.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.users"
)

# ============================================================
# EVENTS
# ============================================================

events = (
    spark.range(100000)
    .withColumn("user_id",(F.rand()*5000+1).cast("int"))
    .withColumn(
        "event_time",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY") * (F.rand()*365)
    )
    .withColumn(
        "event_type",
        F.expr("""
        CASE
            WHEN rand() < 0.25 THEN 'login'
            WHEN rand() < 0.50 THEN 'view'
            WHEN rand() < 0.75 THEN 'click'
            ELSE 'purchase'
        END
        """)
    )
)

events.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.events"
)

# ============================================================
# CUSTOMERS
# ============================================================

customers = (
    spark.range(1,10001)
    .withColumnRenamed("id","customer_id")
    .withColumn("name",F.concat(F.lit("Customer_"),F.col("customer_id")))
    .withColumn("phone",F.concat(F.lit("07"),F.lpad((F.rand()*100000000).cast("int"),8,"0")))
    .withColumn("email",F.concat(F.lit("customer"),F.col("customer_id"),F.lit("@test.com")))
    .withColumn("updated_at",F.current_timestamp())
)

customers.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.customers"
)

# ============================================================
# DUPLICATE CUSTOMER RECORDS
# ============================================================

duplicates = (
    customers.limit(200)
    .withColumn("customer_id",F.col("customer_id")+100000)
    .withColumn("updated_at",F.current_timestamp())
)

duplicates.write.mode("append").saveAsTable(
    f"{CATALOG}.{SCHEMA}.customers"
)

# ============================================================
# PRODUCTS
# ============================================================

products = (
    spark.range(1,1001)
    .withColumnRenamed("id","product_id")
    .withColumn(
        "category",
        F.concat(F.lit("Category_"),(F.rand()*20).cast("int"))
    )
    .withColumn(
        "price",
        F.round(F.rand()*500+5,2)
    )
)

products.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.products"
)

# ============================================================
# ORDERS
# ============================================================

orders = (
    spark.range(1,50001)
    .withColumnRenamed("id","order_id")
    .withColumn("customer_id",(F.rand()*10000+1).cast("int"))
    .withColumn(
        "order_time",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*730)
    )
    .withColumn(
        "amount",
        F.round(F.rand()*2000+20,2)
    )
)

orders.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.orders"
)

# ============================================================
# ORDER ITEMS
# ============================================================

order_items = (
    spark.range(1,200001)
    .withColumn("order_id",(F.rand()*50000+1).cast("int"))
    .withColumn("product_id",(F.rand()*1000+1).cast("int"))
    .withColumn("qty",(F.rand()*5+1).cast("int"))
)

order_items.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.order_items"
)

# ============================================================
# TRANSACTIONS
# ============================================================

transactions = (
    spark.range(1,100001)
    .withColumnRenamed("id","transaction_id")
    .withColumn("account_id",(F.rand()*5000+1).cast("int"))
    .withColumn(
        "ts",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn(
        "amount",
        F.round((F.rand()-0.45)*1000,2)
    )
)

transactions.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.transactions"
)

# ============================================================
# SESSIONS
# ============================================================

sessions = (
    spark.range(1,30001)
    .withColumnRenamed("id","session_id")
    .withColumn("user_id",(F.rand()*5000+1).cast("int"))
    .withColumn(
        "start_time",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn(
        "duration_minutes",
        (F.rand()*300+1).cast("int")
    )
)

sessions = sessions.withColumn(
    "end_time",
    F.expr("start_time + INTERVAL duration_minutes MINUTES")
).drop("duration_minutes")

sessions.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.sessions"
)

# ============================================================
# PAGEVIEWS
# ============================================================

pageviews = (
    spark.range(1,200000)
    .withColumn("user_id",(F.rand()*5000+1).cast("int"))
    .withColumn(
        "ts",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn(
        "page",
        F.expr("""
        CASE
            WHEN rand() < 0.3 THEN 'homepage'
            WHEN rand() < 0.6 THEN 'product'
            ELSE 'checkout'
        END
        """)
    )
)

pageviews.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.pageviews"
)

# ============================================================
# INVENTORY
# ============================================================

inventory = (
    spark.range(1,5001)
    .withColumn("product_id",(F.rand()*1000+1).cast("int"))
    .withColumn("warehouse_id",(F.rand()*10+1).cast("int"))
    .withColumn("on_hand",(F.rand()*1000+10).cast("int"))
)

inventory.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.inventory"
)

# ============================================================
# SALES
# ============================================================

sales = (
    spark.range(1,100000)
    .withColumn("product_id",(F.rand()*1000+1).cast("int"))
    .withColumn(
        "ts",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn("qty",(F.rand()*10+1).cast("int"))
)

sales.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.sales"
)

# ============================================================
# SUBSCRIPTIONS
# ============================================================

subscriptions = (
    spark.range(1,30001)
    .withColumn("user_id",(F.rand()*5000+1).cast("int"))
    .withColumn(
        "start_date",
        F.date_sub(F.current_date(),(F.rand()*365).cast("int"))
    )
)

subscriptions = subscriptions.withColumn(
    "end_date",
    F.date_add(
        F.col("start_date"),
        (F.rand()*120+10).cast("int")
    )
)

subscriptions.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.subscriptions"
)

# ============================================================
# SHIPMENTS
# ============================================================

shipments = (
    spark.range(1,30001)
    .withColumnRenamed("id","order_id")
    .withColumn(
        "carrier",
        F.expr("""
        CASE
            WHEN rand() < 0.33 THEN 'DHL'
            WHEN rand() < 0.66 THEN 'UPS'
            ELSE 'RoyalMail'
        END
        """)
    )
    .withColumn(
        "shipped_at",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*180)
    )
)

shipments.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.shipments"
)

# ============================================================
# DELIVERIES
# ============================================================

deliveries = shipments.withColumn(
    "delivered_at",
    F.expr("shipped_at + INTERVAL 1 DAY") +
    F.expr("INTERVAL int(rand()*7) DAY")
).select("order_id","delivered_at")

deliveries.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.deliveries"
)

# ============================================================
# TICKETS
# ============================================================

tickets = (
    spark.range(1,20001)
    .withColumnRenamed("id","ticket_id")
    .withColumn("agent_id",(F.rand()*100+1).cast("int"))
    .withColumn(
        "opened_at",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*180)
    )
)

tickets = tickets.withColumn(
    "closed_at",
    F.expr("opened_at + INTERVAL 1 DAY") +
    F.expr("INTERVAL int(rand()*10) DAY")
)

tickets.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.tickets"
)

# ============================================================
# CHANGELOG TABLE
# ============================================================

customer_changelog = (
    spark.range(1,50000)
    .withColumn("customer_id",(F.rand()*10000+1).cast("int"))
    .withColumn(
        "change_time",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn(
        "city",
        F.when(F.rand()>0.5,"London")
         .otherwise(None)
    )
)

customer_changelog.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.customer_changelog"
)

# ============================================================
# INGESTION EVENTS
# ============================================================

ingestion_events = (
    spark.range(1,50000)
    .withColumn(
        "business_key",
        (F.rand()*10000).cast("int")
    )
    .withColumn(
        "event_time",
        F.current_timestamp() -
        F.expr("INTERVAL 1 DAY")*(F.rand()*365)
    )
    .withColumn(
        "ingestion_time",
        F.current_timestamp()
    )
)

ingestion_events.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.ingestion_events"
)

print("===================================================")
print(f"Created all interview prep tables in:")
print(f"{CATALOG}.{SCHEMA}")
print("===================================================")

spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(100,False)